# 29 - Maquinas de vectores de soporte: margen, `C` y kernels

Una maquina de vectores de soporte, o **SVM** por sus siglas en ingles, busca una frontera de decision con una idea geometrica:
separar clases dejando el margen mas amplio posible entre ellas.

La frontera no depende de todos los ejemplos por igual. Algunos puntos cercanos al limite, llamados **vectores de soporte**, son especialmente importantes.
Cuando los datos no se pueden separar con una recta, los **kernels** permiten construir fronteras no lineales sin perder la idea central del algoritmo.

La meta de este notebook es entender el modelo por dentro y practicarlo con visualizaciones, no solo llamar a `sklearn`.

Al terminar deberias poder:

1. Explicar que son hiperplano, margen y vectores de soporte.
2. Observar por que mover un vector de soporte suele importar mas que mover un punto lejano.
3. Interpretar el parametro de penalizacion `C`.
4. Explicar como una transformacion a mas dimensiones puede volver lineal un problema no lineal.
5. Comparar fronteras producidas por kernels lineal, polinomial, RBF y sigmoide.
6. Entender el papel de `gamma` en kernels como RBF.
7. Entrenar y evaluar una SVM dentro de un flujo reproducible con escalado y validacion.

## Ruta de la sesion

1. Intuicion: separar con el mayor margen posible.
2. Vectores de soporte: los puntos que sostienen la frontera.
3. Margen suave y parametro `C`.
4. Transformar datos a una dimension mayor.
5. Kernel trick y comparacion de kernels.
6. Parametro `gamma` y complejidad local.
7. Escalado, validacion y evaluacion.
8. Ventajas, limites y ejercicios para pensar.

## 1) Preparacion

Este notebook usa solo librerias comunes del curso: `numpy`, `pandas`, `matplotlib` y `scikit-learn`.
Los datos son autocontenidos: se generan o vienen incluidos en `scikit-learn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, make_blobs, make_moons
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 29

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["font.size"] = 11

## 2) Intuicion: separar con el mayor margen posible

Para dos clases, una SVM lineal busca una frontera:

$$
w^T x + b = 0
$$

En dos dimensiones, esa frontera es una recta. En tres dimensiones es un plano.
En dimensiones mayores usamos la palabra **hiperplano**.

Hay muchas rectas capaces de separar algunos conjuntos de datos. La SVM prefiere una con margen amplio:

- la linea central es la frontera de decision;
- las lineas paralelas $w^T x + b = -1$ y $w^T x + b = 1$ delimitan el margen;
- el ancho total del margen es $\frac{2}{||w||}$;
- maximizar el margen ayuda a que la regla no dependa demasiado de pequenas variaciones.

Importante: margen amplio no significa perfeccion garantizada. Es una preferencia geometrica que debe validarse con datos no vistos.

## 3) Primera frontera lineal

Construiremos un problema pequeno y separable. Esto permite ver la frontera, los margenes y los vectores de soporte.
Usaremos `C=100` para aproximarnos visualmente a una separacion estricta.

In [ ]:
X_linear = np.array([
    [-3.8, -0.8],
    [-3.2,  0.9],
    [-2.7, -1.4],
    [-2.3,  0.1],
    [-1.8, -0.7],
    [-1.6,  0.8],
    [ 1.4, -0.6],
    [ 1.7,  0.9],
    [ 2.2, -1.3],
    [ 2.6,  0.2],
    [ 3.2,  1.2],
    [ 3.8, -0.5],
])
y_linear = np.array([0] * 6 + [1] * 6)


def plot_svc_regions(
    model,
    X,
    y,
    ax=None,
    title="",
    show_margins=True,
    highlight_support=True,
    xlim=None,
    ylim=None,
):
    """Dibuja regiones, frontera y vectores de soporte para una SVC binaria."""
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 5))

    if xlim is None:
        xlim = (X[:, 0].min() - 0.8, X[:, 0].max() + 0.8)
    if ylim is None:
        ylim = (X[:, 1].min() - 0.8, X[:, 1].max() + 0.8)

    xx, yy = np.meshgrid(
        np.linspace(xlim[0], xlim[1], 450),
        np.linspace(ylim[0], ylim[1], 450),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    predictions = model.predict(grid).reshape(xx.shape)
    decision = model.decision_function(grid).reshape(xx.shape)

    ax.contourf(
        xx,
        yy,
        predictions,
        levels=[-0.5, 0.5, 1.5],
        cmap="coolwarm",
        alpha=0.18,
    )

    if show_margins:
        ax.contour(
            xx,
            yy,
            decision,
            levels=[-1, 0, 1],
            colors=["gray", "black", "gray"],
            linestyles=["--", "-", "--"],
            linewidths=[1.1, 1.8, 1.1],
        )
    else:
        ax.contour(
            xx,
            yy,
            decision,
            levels=[0],
            colors="black",
            linewidths=1.8,
        )

    for class_id, color in zip([0, 1], ["#3b6fb6", "#c94c4c"]):
        mask = y == class_id
        ax.scatter(
            X[mask, 0],
            X[mask, 1],
            label=f"clase {class_id}",
            c=color,
            edgecolor="black",
            linewidth=0.5,
            s=55,
            alpha=0.9,
        )

    if highlight_support:
        ax.scatter(
            model.support_vectors_[:, 0],
            model.support_vectors_[:, 1],
            s=190,
            facecolors="none",
            edgecolors="black",
            linewidths=1.5,
            label="vectores de soporte",
        )

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
    return ax


linear_svm = SVC(kernel="linear", C=100)
linear_svm.fit(X_linear, y_linear)

fig, ax = plt.subplots(figsize=(8, 5.5))
plot_svc_regions(
    linear_svm,
    X_linear,
    y_linear,
    ax=ax,
    title="SVM lineal: frontera, margenes y vectores de soporte",
)
plt.show()

In [ ]:
support_table = pd.DataFrame(
    X_linear[linear_svm.support_],
    columns=["x1", "x2"],
    index=linear_svm.support_,
)
support_table["clase"] = y_linear[linear_svm.support_]
support_table.index.name = "indice_original"
support_table

## 4) Que son los vectores de soporte

Los puntos rodeados por un circulo son los **vectores de soporte**.
Son ejemplos cercanos al margen o dentro de el. La posicion de la frontera depende directamente de ellos.

La funcion de decision puede escribirse como:

$$
f(x) = \sum_{i \in SV} \alpha_i y_i K(x_i, x) + b
$$

donde:

- `SV` es el conjunto de vectores de soporte;
- $\alpha_i$ mide el peso aprendido para cada vector de soporte;
- $y_i$ es su clase;
- $K(x_i, x)$ es el kernel;
- $b$ desplaza la frontera.

Los ejemplos suficientemente lejanos suelen tener $\alpha_i = 0$. Por eso no aparecen en la suma final.
Esto no significa que sean inutiles: ayudaron a confirmar que existe separacion. Significa que, una vez ajustado el modelo, no sostienen la frontera.

## 5) Mover un punto lejano frente a mover un vector de soporte

Vamos a comparar tres situaciones:

1. datos originales;
2. movemos un punto muy lejano sin acercarlo al margen;
3. movemos hacia el centro uno de los vectores de soporte.

La expectativa es que el segundo cambio tenga poco o ningun efecto, mientras que el tercero modifique la frontera.

In [ ]:
def fit_linear_svm(X, y, C=100):
    model = SVC(kernel="linear", C=C)
    return model.fit(X, y)


base_model = fit_linear_svm(X_linear, y_linear)

far_index = int(np.argmax(np.abs(base_model.decision_function(X_linear))))
support_index = int(base_model.support_[0])

X_far_moved = X_linear.copy()
X_far_moved[far_index] += np.array([0.0, 0.65])

X_support_moved = X_linear.copy()
X_support_moved[support_index, 0] -= np.sign(X_support_moved[support_index, 0]) * 0.85

comparison = [
    ("Datos originales", X_linear, base_model, None),
    ("Movemos punto lejano", X_far_moved, fit_linear_svm(X_far_moved, y_linear), far_index),
    ("Movemos vector de soporte", X_support_moved, fit_linear_svm(X_support_moved, y_linear), support_index),
]

common_xlim = (-4.8, 4.8)
common_ylim = (-2.2, 2.2)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), constrained_layout=True)

for ax, (title, X_variant, model, moved_index) in zip(axes, comparison):
    plot_svc_regions(
        model,
        X_variant,
        y_linear,
        ax=ax,
        title=title,
        xlim=common_xlim,
        ylim=common_ylim,
    )
    if moved_index is not None:
        ax.scatter(
            X_variant[moved_index, 0],
            X_variant[moved_index, 1],
            marker="*",
            s=260,
            c="gold",
            edgecolor="black",
            linewidth=0.8,
            label="punto movido",
        )
        ax.legend(loc="best", fontsize=8)

plt.show()

In [ ]:
rows = []
for title, _, model, _ in comparison:
    rows.append({
        "escenario": title,
        "w1": model.coef_[0, 0],
        "w2": model.coef_[0, 1],
        "b": model.intercept_[0],
        "n_vectores_soporte": model.n_support_.sum(),
    })

pd.DataFrame(rows).round(4)

Lectura esperada:

- mover un punto lejano puede dejar practicamente igual la frontera si el punto continua fuera del margen;
- mover un vector de soporte cambia la restriccion geometrica mas exigente y puede rotar o desplazar la frontera;
- si un punto lejano se acerca lo suficiente, puede convertirse en vector de soporte y entonces si afectar el modelo;
- si eliminamos o modificamos datos, siempre debemos volver a entrenar: la frontera aprendida no se actualiza sola.

## 6) Cuando no existe una separacion perfecta: margen suave

Los datos reales suelen tener ruido, valores atipicos o clases que se traslapan.
Una SVM practica permite que ciertos ejemplos entren al margen o incluso queden del lado incorrecto.

Para cada ejemplo aparece una holgura $\xi_i$:

$$
\min_{w, b, \xi} \frac{1}{2} ||w||^2 + C \sum_i \xi_i
$$

sujeto a:

$$
y_i(w^T x_i + b) \geq 1 - \xi_i, \qquad \xi_i \geq 0
$$

Aqui compiten dos objetivos:

1. $\frac{1}{2} ||w||^2$: preferir un margen amplio;
2. $C \sum_i \xi_i$: penalizar violaciones al margen.

El parametro `C` decide cuanto pesa el segundo objetivo.

### 6.1 Papel del parametro `C`

| Valor de `C` | Que prioriza | Efecto frecuente |
|---|---|---|
| `C` pequeno | Regularizacion y margen mas tolerante | Acepta mas violaciones; puede subajustar. |
| `C` grande | Penalizar con fuerza las violaciones | Intenta ajustar mas los datos; puede depender demasiado del ruido. |

`C` no significa "numero de clases", ni garantiza que un valor alto sea mejor.
Debe elegirse con validacion.

In [ ]:
X_c, y_c = make_blobs(
    n_samples=150,
    centers=[(-1.0, -0.8), (1.0, 0.8)],
    cluster_std=1.15,
    random_state=RANDOM_STATE,
)

C_values = [0.05, 1, 100]
rows = []
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), constrained_layout=True)

for ax, C in zip(axes, C_values):
    model = SVC(kernel="linear", C=C)
    model.fit(X_c, y_c)

    signed_distance = (2 * y_c - 1) * model.decision_function(X_c)
    margin_violations = int(np.sum(signed_distance < 1 - 1e-9))
    errors = int(np.sum(model.predict(X_c) != y_c))

    rows.append({
        "C": C,
        "accuracy_train": model.score(X_c, y_c),
        "violaciones_margen": margin_violations,
        "errores_clasificacion": errors,
        "n_vectores_soporte": model.n_support_.sum(),
    })

    plot_svc_regions(
        model,
        X_c,
        y_c,
        ax=ax,
        title=f"C={C} | violaciones={margin_violations}",
    )

plt.show()
pd.DataFrame(rows).round(3)

La tabla distingue dos ideas:

- **violacion al margen**: el punto esta entre las lineas punteadas o del lado incorrecto;
- **error de clasificacion**: el punto queda del lado incorrecto de la frontera central.

Un ejemplo puede violar el margen y aun asi clasificarse correctamente.
Observa tambien que muchos puntos pueden convertirse en vectores de soporte cuando el problema es dificil.

## 7) De una dimension baja a una dimension mayor

Considera un problema en una sola dimension:

- los puntos del centro pertenecen a la clase `0`;
- los extremos pertenecen a la clase `1`.

En el eje original no existe un unico corte que separe las clases.
Pero podemos transformar cada dato:

$$
\phi(x) = (x, x^2)
$$

Pasamos de **una dimension** a **dos dimensiones**.
En el nuevo espacio, una linea horizontal puede separar centro y extremos.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
x_1d = np.linspace(-2.8, 2.8, 140)
y_1d = (np.abs(x_1d) > 1.45).astype(int)
phi_x = np.c_[x_1d, x_1d ** 2]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)

axes[0].scatter(
    x_1d,
    rng.normal(0, 0.025, size=len(x_1d)),
    c=y_1d,
    cmap="coolwarm",
    edgecolor="black",
    linewidth=0.25,
)
axes[0].axvline(-1.45, color="black", linestyle="--", linewidth=1)
axes[0].axvline(1.45, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Espacio original: una dimension")
axes[0].set_xlabel("x")
axes[0].set_yticks([])

axes[1].scatter(
    phi_x[:, 0],
    phi_x[:, 1],
    c=y_1d,
    cmap="coolwarm",
    edgecolor="black",
    linewidth=0.25,
)
axes[1].axhline(1.45 ** 2, color="black", linestyle="--", label="separacion lineal")
axes[1].set_title(r"Espacio transformado: $\phi(x) = (x, x^2)$")
axes[1].set_xlabel("x")
axes[1].set_ylabel(r"$x^2$")
axes[1].legend()

plt.show()

La frontera es lineal en el espacio transformado, pero al volver al eje original se convierte en una regla no lineal:

$$
x^2 > 1.45^2
$$

Esto produce dos zonas exteriores en lugar de un solo corte.

La idea se generaliza: una frontera sencilla en un espacio de mayor dimension puede verse curva en el espacio original.

## 8) Kernel trick: comparar similitudes sin construir todas las variables

Transformar los datos explicitamente puede ser costoso si aparecen muchas dimensiones.
Un **kernel** calcula productos internos en el espacio transformado sin construir cada nueva variable.

Ejemplo: para dos variables y un kernel polinomial cuadratico:

$$
K(x, z) = (x^T z)^2
$$

existe una transformacion:

$$
\phi(x) = (x_1^2, \sqrt{2}x_1x_2, x_2^2)
$$

tal que:

$$
K(x, z) = \phi(x)^T \phi(z)
$$

En lugar de crear manualmente cuadrados e interacciones, el kernel permite trabajar con la similitud equivalente.
Para RBF, el espacio asociado puede entenderse como uno de dimension muy alta, incluso infinita.

### 8.1 Kernels comunes

| Kernel | Idea | Uso intuitivo |
|---|---|---|
| `linear` | $K(x,z) = x^Tz$ | Frontera lineal; buen punto de partida. |
| `poly` | $(\gamma x^Tz + r)^d$ | Interacciones polinomiales; controla grado `d`. |
| `rbf` | $\exp(-\gamma ||x-z||^2)$ | Similitud local; fronteras curvas flexibles. |
| `sigmoid` | $\tanh(\gamma x^Tz + r)$ | Relacion historica con neuronas; no siempre funciona bien. |

Un kernel mas flexible no es automaticamente mejor.
La geometria del problema y la validacion deben justificar la eleccion.

## 9) Como cambia la frontera segun el kernel

Usaremos datos con forma de dos lunas. Una recta dificilmente puede seguir esta geometria.
Para comparar de forma justa, escalaremos las dos variables antes de entrenar.

In [ ]:
X_kernels, y_kernels = make_moons(
    n_samples=360,
    noise=0.24,
    random_state=RANDOM_STATE,
)
X_train_k, X_test_k, y_train_k, y_test_k = train_test_split(
    X_kernels,
    y_kernels,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y_kernels,
)

scaler_k = StandardScaler()
X_train_k_sc = scaler_k.fit_transform(X_train_k)
X_test_k_sc = scaler_k.transform(X_test_k)

kernel_specs = [
    ("lineal", {"kernel": "linear", "C": 1}),
    ("polinomial grado 3", {"kernel": "poly", "degree": 3, "coef0": 1, "C": 1}),
    ("RBF", {"kernel": "rbf", "gamma": "scale", "C": 1}),
    ("sigmoide", {"kernel": "sigmoid", "gamma": "scale", "C": 1}),
]

rows = []
fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)

for ax, (name, params) in zip(axes.ravel(), kernel_specs):
    model = SVC(**params)
    model.fit(X_train_k_sc, y_train_k)

    train_accuracy = model.score(X_train_k_sc, y_train_k)
    test_accuracy = model.score(X_test_k_sc, y_test_k)
    rows.append({
        "kernel": name,
        "accuracy_train": train_accuracy,
        "accuracy_test": test_accuracy,
        "n_vectores_soporte": model.n_support_.sum(),
    })

    plot_svc_regions(
        model,
        X_train_k_sc,
        y_train_k,
        ax=ax,
        title=f"{name} | accuracy test={test_accuracy:.2f}",
        show_margins=False,
    )

plt.show()
kernel_results = pd.DataFrame(rows).sort_values("accuracy_test", ascending=False)
kernel_results.round(3)

Preguntas utiles para leer las graficas:

- Puede una recta representar la geometria de las lunas?
- Que curvatura agrega el kernel polinomial?
- RBF sigue la estructura sin crear demasiadas islas?
- El kernel sigmoide aporta algo en este conjunto?
- La mejor frontera visual coincide con el mejor resultado en datos de prueba?

La grafica ayuda a formar hipotesis. La metrica en datos no vistos ayuda a decidir.

## 10) El parametro `gamma`: alcance de influencia en RBF

Para el kernel RBF:

$$
K(x, z) = \exp(-\gamma ||x-z||^2)
$$

`gamma` controla que tan rapido cae la similitud al alejarnos de un ejemplo:

| Valor de `gamma` | Alcance de cada ejemplo | Efecto frecuente |
|---|---|---|
| Pequeno | Amplio | Frontera mas suave; puede subajustar. |
| Grande | Local | Frontera mas detallada; puede sobreajustar. |

`C` y `gamma` interactuan. Por eso conviene explorarlos juntos mediante validacion.

In [ ]:
gamma_values = [0.1, 1, 20]
rows = []
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), constrained_layout=True)

for ax, gamma in zip(axes, gamma_values):
    model = SVC(kernel="rbf", C=10, gamma=gamma)
    model.fit(X_train_k_sc, y_train_k)

    train_accuracy = model.score(X_train_k_sc, y_train_k)
    test_accuracy = model.score(X_test_k_sc, y_test_k)
    rows.append({
        "gamma": gamma,
        "accuracy_train": train_accuracy,
        "accuracy_test": test_accuracy,
        "n_vectores_soporte": model.n_support_.sum(),
    })

    plot_svc_regions(
        model,
        X_train_k_sc,
        y_train_k,
        ax=ax,
        title=f"RBF | gamma={gamma} | test={test_accuracy:.2f}",
        show_margins=False,
    )

plt.show()
pd.DataFrame(rows).round(3)

Observa el equilibrio:

- con `gamma` pequeno, muchos puntos se perciben como parecidos y la frontera se suaviza;
- con `gamma` grande, la influencia se vuelve local y aparecen detalles mas finos;
- un accuracy de entrenamiento alto no basta para elegir el modelo;
- una frontera excesivamente irregular es una senal para revisar generalizacion.

## 11) Escalar variables no es opcional

SVM usa distancias y productos internos.
Si una variable toma valores entre `0` y `100000`, mientras otra varia entre `0` y `1`, la primera puede dominar la geometria aunque no sea mas importante.

`StandardScaler` transforma cada variable usando estadisticas del conjunto de entrenamiento:

$$
z = \frac{x - \mu_{train}}{\sigma_{train}}
$$

Regla practica:

1. divide train y test;
2. ajusta el escalador solo con train;
3. transforma train y test con ese escalador;
4. entrena la SVM;
5. usa `Pipeline` para automatizar el orden y evitar fuga de informacion.

## 12) Flujo completo: escalado, validacion y evaluacion

Usaremos el dataset de cancer de mama incluido en `scikit-learn`.
El objetivo es predecir si un tumor es maligno o benigno a partir de mediciones numericas.

El conjunto de prueba se reserva para la evaluacion final.
La seleccion de kernel, `C` y `gamma` se hace mediante validacion cruzada dentro del conjunto de entrenamiento.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X_cancer = cancer.data
y_cancer = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X_cancer,
    y_cancer,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_cancer,
)

print("Forma de train:", X_train.shape)
print("Forma de test:", X_test.shape)
print("Clases:", dict(enumerate(cancer.target_names)))

ranges = (X_train.max() - X_train.min()).sort_values(ascending=False)
pd.DataFrame({"rango_en_train": ranges}).head(8)

In [ ]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC()),
])

param_grid = [
    {
        "svc__kernel": ["linear"],
        "svc__C": [0.01, 0.1, 1, 10],
    },
    {
        "svc__kernel": ["rbf"],
        "svc__C": [0.1, 1, 10, 100],
        "svc__gamma": ["scale", 0.01, 0.1, 1],
    },
]

search = GridSearchCV(
    svm_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=1,
)
search.fit(X_train, y_train)

print("Mejores parametros:", search.best_params_)
print("Accuracy promedio en validacion:", round(search.best_score_, 4))

cv_results = (
    pd.DataFrame(search.cv_results_)
    .sort_values("rank_test_score")
    [["rank_test_score", "mean_test_score", "std_test_score", "params"]]
    .head(8)
)
cv_results

In [ ]:
best_svm = search.best_estimator_
y_pred = best_svm.predict(X_test)

print("Accuracy final en test:", round(accuracy_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_estimator(
    best_svm,
    X_test,
    y_test,
    display_labels=cancer.target_names,
    cmap="Blues",
    ax=ax,
)
ax.set_title("Matriz de confusion en prueba")
plt.show()

fitted_svc = best_svm.named_steps["svc"]
print("Vectores de soporte por clase:", fitted_svc.n_support_)
print("Total de vectores de soporte:", fitted_svc.n_support_.sum())

Una matriz de confusion obliga a mirar mas alla de accuracy.

Preguntas utiles:

- Que significa un falso negativo en este problema?
- Tiene el mismo costo que un falso positivo?
- Accuracy es suficiente para decidir el despliegue?
- Debemos optimizar otra metrica, ajustar pesos de clase o revisar el umbral operativo?

Un modelo predictivo no reemplaza la definicion responsable del problema.

## 13) Detalles utiles que conviene recordar

### 13.1 Multiclase

La explicacion geometrica anterior usa dos clases.
`SVC` puede trabajar con mas clases construyendo varios clasificadores binarios internamente con estrategia uno contra uno.

### 13.2 Probabilidades

La distancia firmada a la frontera viene de `decision_function`.
No es una probabilidad.
`SVC(probability=True)` puede estimar probabilidades mediante calibracion adicional, pero aumenta el costo de entrenamiento.

### 13.3 Peso de clases

Si una clase poco frecuente es importante, revisa `class_weight="balanced"` o pesos definidos por el problema.
La eleccion debe evaluarse con metricas adecuadas, no asumirse correcta por defecto.

### 13.4 Numero de vectores de soporte

Los vectores de soporte son ejemplos influyentes para la frontera.
No deben interpretarse automaticamente como errores, anomalias o casos que deban eliminarse.

## 14) Ventajas, limites y usos razonables

Ventajas:

- Idea geometrica clara: margen y similitud.
- Buen rendimiento en muchos problemas pequenos o medianos.
- Kernels permiten fronteras no lineales.
- Puede funcionar bien cuando hay muchas variables.

Limites:

- El escalado de variables es importante.
- Elegir kernel, `C` y `gamma` requiere validacion.
- Entrenar puede ser costoso con conjuntos muy grandes.
- Una frontera con kernel suele ser menos interpretable que una regla simple.
- No produce probabilidades calibradas de forma directa.

Usos razonables:

- Modelo base lineal para clasificacion.
- Problemas medianos con fronteras no lineales justificadas.
- Comparacion contra k-NN, arboles y modelos lineales.
- Casos donde importa estudiar ejemplos cercanos a la frontera.

## 15) Checklist de aprendizaje

Antes de pasar a los ejercicios, intenta responder sin ver la teoria:

1. Que diferencia hay entre frontera de decision y margen?
2. Por que mover un punto lejano puede no cambiar una SVM entrenada?
3. Que ocurre frecuentemente al aumentar mucho `C`?
4. Como puede una transformacion a mas dimensiones facilitar la separacion?
5. Que evita calcular explicitamente el kernel trick?
6. Que diferencia conceptual hay entre `C` y `gamma`?
7. Por que debemos escalar dentro de un `Pipeline`?
8. Por que un vector de soporte no es automaticamente un dato incorrecto?

## 16) Ejercicios guiados para pensar y practicar

En cada ejercicio:

1. escribe primero una hipotesis en palabras;
2. define que evidencia podria contradecirla;
3. ejecuta un experimento;
4. redacta una conclusion corta.

La meta no es copiar codigo del notebook, sino justificar decisiones.

### Ejercicio 1: cuando un punto empieza a importar

Usa `X_linear` y `y_linear`.

1. Elige un punto que no sea vector de soporte.
2. Predice que ocurrira si lo acercas gradualmente al margen en cinco pasos.
3. Reentrena una SVM lineal despues de cada movimiento.
4. Registra cuando se convierte en vector de soporte.
5. Explica si el cambio en la frontera ocurre de golpe o gradualmente.

Guia: puedes usar `model.support_`, `model.coef_` y `model.intercept_`.

In [ ]:
# Ejercicio 1
# 1. Escoge un indice fuera de base_model.support_.
# 2. Construye cinco posiciones cada vez mas cercanas al centro.
# 3. Reentrena y guarda: paso, posicion, es_soporte, w1, w2 y b.
# 4. Convierte tus resultados a DataFrame y explica el patron observado.

### Ejercicio 2: `C` como decision de regularizacion

Usa `X_c` y `y_c`.

1. Antes de ejecutar, ordena `C = [0.001, 0.01, 0.1, 1, 10, 1000]` desde el que esperas que tolere mas violaciones hasta el que tolere menos.
2. Divide los datos en train y test.
3. Para cada `C`, registra accuracy de train, accuracy de test, errores y numero de vectores de soporte.
4. Elige un valor sin usar solamente el accuracy de train.
5. Explica que evidencia te haria cambiar de eleccion.

In [ ]:
# Ejercicio 2
C_candidates = [0.001, 0.01, 0.1, 1, 10, 1000]

# TODO: divide X_c, y_c de forma estratificada.
# TODO: entrena una SVC lineal por cada C.
# TODO: crea una tabla y argumenta tu eleccion.

### Ejercicio 3: disenar una transformacion explicita

En la seccion 7 usamos $\phi(x) = (x, x^2)$.

Ahora imagina puntos en dos dimensiones distribuidos como circulos concentricos.

1. Por que una recta en `(x1, x2)` no suele bastar?
2. Propone una nueva variable construida a partir de `x1` y `x2`.
3. Dibuja esa variable para ambas clases.
4. Explica como se relaciona tu propuesta con el kernel RBF.
5. Senala una diferencia entre crear tu variable manualmente y usar un kernel.

In [ ]:
# Ejercicio 3
# Pista conceptual: piensa en distancia al origen.
# Puedes importar make_circles desde sklearn.datasets para generar datos.
# No entrenes primero: dibuja tu transformacion y decide si separa las clases.

### Ejercicio 4: distinguir el papel de `C` y `gamma`

Usa los datos escalados de lunas: `X_train_k_sc`, `X_test_k_sc`, `y_train_k`, `y_test_k`.

1. Construye una tabla para todas las combinaciones de:
   - `C = [0.1, 1, 10, 100]`
   - `gamma = [0.01, 0.1, 1, 10, 100]`
2. Registra accuracy de train y test.
3. Identifica una combinacion que subajuste y otra que parezca sobreajustar.
4. Explica por separado que papel jugo cada parametro.
5. Dibuja solo tres fronteras que respalden tu argumento.

In [ ]:
# Ejercicio 4
C_candidates = [0.1, 1, 10, 100]
gamma_candidates = [0.01, 0.1, 1, 10, 100]

# TODO: construye una tabla con un doble ciclo.
# TODO: selecciona tres casos informativos, no solamente el de mejor accuracy.

### Ejercicio 5: escalado y fuga de informacion

Usa `X_cancer` y `y_cancer`.

Compara tres flujos:

1. `SVC` sin escalado;
2. escalado antes de dividir train y test;
3. `Pipeline([("scaler", StandardScaler()), ("svc", SVC(...))])`.

Responde:

1. Cual es metodologicamente correcto?
2. Por que el segundo flujo filtra informacion aunque no use las etiquetas?
3. La diferencia numerica siempre sera grande?
4. Por que una fuga pequena sigue siendo un problema?

In [ ]:
# Ejercicio 5
# TODO: implementa los tres flujos con el mismo random_state.
# TODO: compara metricas y separa la conclusion numerica de la metodologica.

### Ejercicio 6: evaluar una decision sensible

Imagina que una SVM apoya una revision medica prioritaria.

1. Que error parece mas grave: falso positivo o falso negativo?
2. Que metrica priorizarias ademas de accuracy?
3. Como estudiarias el efecto de `class_weight`?
4. Que informacion adicional necesitarias antes de recomendar despliegue?
5. Por que no basta con decir que la SVM tuvo buen accuracy?

Redacta una recomendacion de maximo ocho lineas para una persona no tecnica.

In [ ]:
# Ejercicio 6
# Escribe aqui tu recomendacion. Separa:
# - evidencia disponible;
# - riesgos;
# - siguiente validacion necesaria.

## 17) Cierre

Una SVM combina tres ideas:

1. buscar una frontera con margen amplio;
2. permitir errores controlados mediante `C`;
3. usar kernels para expresar fronteras no lineales como separaciones lineales en espacios transformados.

Los vectores de soporte muestran donde se concentra la dificultad del problema.
Pero una buena practica no termina al dibujar una frontera: exige escalado correcto, validacion, metricas adecuadas y una interpretacion responsable del costo de los errores.